In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
patients_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/Volumes/workspace/data/aws_etl/data/patient_details.csv")
display(patients_df)
vitals_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/Volumes/workspace/data/aws_etl/data/patient_vitals.csv")
display(vitals_df)

devices_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/Volumes/workspace/data/aws_etl/data/device_logs.csv")
display(devices_df)

patient_id,name,age,gender,city,disease
P1001,Rahul Sharma,45,Male,Delhi,Diabetes
P1002,Priya Verma,52,Female,Mumbai,Hypertension
P1003,Amit Singh,60,Male,Pune,Heart Disease
P1004,Neha Kapoor,29,Female,Delhi,Asthma
P1005,Rohit Jain,48,Male,Jaipur,Diabetes


patient_id,timestamp,heart_rate,blood_pressure,oxygen_level,temperature
P1001,1/10/2025 8:00,72,120/80,98,98.6
P1002,1/10/2025 8:05,95,140/90,96,99.1
P1003,1/10/2025 8:10,45,90/60,88,101.2
P1004,1/10/2025 8:15,null,130/85,97,98.4
P1005,1/10/2025 8:20,110,150/95,null,100.1
P1005,1/10/2025 8:20,110,150/95,null,100.1


device_id,patient_id,device_type,status,last_sync
D101,P1001,SmartWatch,Active,2025-01-10T08:00:00.000Z
D102,P1002,Oximeter,Inactive,2025-01-10T07:50:00.000Z
D103,P1003,HeartMonitor,Active,2025-01-10T08:05:00.000Z


In [0]:

clean_vitals_df = (vitals_df
.dropDuplicates()
.filter(col("patient_id").isNotNull())
.withColumn("heart_rate", when(col("heart_rate").isNull(), lit(0)).otherwise(col("heart_rate")))
.withColumn("oxygen_level", when(col("oxygen_level").isNull(), lit(0)).otherwise(col("oxygen_level")))
.withColumn("timestamp_converted", to_timestamp(col("timestamp"), "M/d/yyyy H:mm"))
.withColumn("monitoring_date", to_date(col("timestamp_converted")))
.withColumn("monitoring_hour", hour(col("timestamp_converted")))
)

#cleans patients and devices DataFrame

clean_patients_df = patients_df.dropDuplicates().filter(col("patient_id").isNotNull())

clean_devices_df = (devices_df.dropDuplicates()
.filter(col("patient_id").isNotNull())
.withColumn("last_sync", to_timestamp(col("last_sync"), "M/d/yyyy H:mm"))
)
display(clean_vitals_df)
display(clean_patients_df)
display(clean_devices_df)

patient_id,timestamp,heart_rate,blood_pressure,oxygen_level,temperature,timestamp_converted,monitoring_date,monitoring_hour
P1005,1/10/2025 8:20,110,150/95,0,100.1,2025-01-10T08:20:00.000Z,2025-01-10,8
P1001,1/10/2025 8:00,72,120/80,98,98.6,2025-01-10T08:00:00.000Z,2025-01-10,8
P1004,1/10/2025 8:15,0,130/85,97,98.4,2025-01-10T08:15:00.000Z,2025-01-10,8
P1003,1/10/2025 8:10,45,90/60,88,101.2,2025-01-10T08:10:00.000Z,2025-01-10,8
P1002,1/10/2025 8:05,95,140/90,96,99.1,2025-01-10T08:05:00.000Z,2025-01-10,8


patient_id,name,age,gender,city,disease
P1002,Priya Verma,52,Female,Mumbai,Hypertension
P1005,Rohit Jain,48,Male,Jaipur,Diabetes
P1004,Neha Kapoor,29,Female,Delhi,Asthma
P1003,Amit Singh,60,Male,Pune,Heart Disease
P1001,Rahul Sharma,45,Male,Delhi,Diabetes


device_id,patient_id,device_type,status,last_sync
D102,P1002,Oximeter,Inactive,2025-01-10T07:50:00.000Z
D103,P1003,HeartMonitor,Active,2025-01-10T08:05:00.000Z
D101,P1001,SmartWatch,Active,2025-01-10T08:00:00.000Z


In [0]:
# Create healthcare business columns
vitals_transformed_df = (
    clean_vitals_df
    .withColumn(
        "health_status",
        when(col("oxygen_level") < 90, "Critical")
        .when(col("temperature") > 100, "Fever")
        .when(col("heart_rate") > 100, "High Risk")
        .otherwise("Normal")
    )
    .withColumn(
        "alert_flag",
        when(
            (col("oxygen_level") < 90) |
            (col("temperature") > 100) |
            (col("heart_rate") > 100),
            lit(True)
        ).otherwise(lit(False))
    )
)

patient_vitals_df = vitals_transformed_df.join(
    clean_patients_df,
    on="patient_id",
    how="inner"
)

# Join with device logs
final_df = patient_vitals_df.join(
    clean_devices_df,
    on="patient_id",
    how="left"
)

# Replace missing device information
final_df = final_df.fillna({
    "device_type": "No Device"
})

# Select final report columns
final_output_df = final_df.select(
    col("patient_id"),
    col("name").alias("patient_name"),
    col("disease"),
    col("heart_rate"),
    col("oxygen_level"),
    col("temperature"),
    col("health_status"),
    col("alert_flag"),
    col("device_type"),
    col("monitoring_date"),
    col("monitoring_hour")
)

display(final_output_df)

patient_id,patient_name,disease,heart_rate,oxygen_level,temperature,health_status,alert_flag,device_type,monitoring_date,monitoring_hour
P1005,Rohit Jain,Diabetes,110,0,100.1,Critical,true,No Device,2025-01-10,8
P1001,Rahul Sharma,Diabetes,72,98,98.6,Normal,false,SmartWatch,2025-01-10,8
P1004,Neha Kapoor,Asthma,0,97,98.4,Normal,false,No Device,2025-01-10,8
P1003,Amit Singh,Heart Disease,45,88,101.2,Critical,true,HeartMonitor,2025-01-10,8
P1002,Priya Verma,Hypertension,95,96,99.1,Normal,false,Oximeter,2025-01-10,8
